In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS projectcatalog.sales_Data

Create Volume

In [0]:
%sql
create volume if not exists projectcatalog.sales_data.data

In [0]:
df_raw_orders = spark.read.csv("/Volumes/projectcatalog/sales_data/data/bronze/retail_dataset.csv", header=True, inferSchema=True)
display(df_raw_orders)

order_id,order_date,customer_id,customer_name,product_id,product_name,category,quantity,price,payment_type,order_status,returned
1001,2025-07-28,C004,Customer_4,P103,Denim Jean,Apparel,1,339.97,card,shipped,no
1002,2025-09-09,C054,Customer_54,P117,USB Cable,Accessories,1,$380.61,cash,completed,no
1003,2025-08-13,C059,Customer_59,P101,T-shirt XL,Apparel,3,278.26,card,completed,no
1004,11-Sep-2025,C013,Customer_13,P107,Charger,Accessories,3,$85.51,upi,completed,no
1005,24/08/2025,C035,Customer_35,P112,Backpack,Accessories,1,343.88,cash,cancelled,no
1006,09-16-2025,C028,Customer_28,P110,Watch,Accessories,3,$75.72,upi,cancelled,no
1007,27/07/2025,C064,Customer_64,P116,keyboard,electronics,2,315.59,upi,cancelled,no
1008,2025-08-10,C069,Customer_69,P103,Denim Jean,Apparel,null,$150.28,upi,delivered,no
1009,06/09/2025,C065,Customer_65,P109,Jeans,Apparel,-1,271.99,cash,cancelled,yes
1010,2025-09-18,C031,Customer_31,P109,jeans,Apparel,1,$47.4,card,cancelled,no


In [0]:
df_raw_customers = spark.read.json("/Volumes/projectcatalog/sales_data/data/bronze/customer_dataset.json")
display(df_raw_customers)


age,city,customer_id,gender,loyalty_tier,signup_date
35.0,Delhi,C001,M,Gold,01-Apr-2020
39.0,Gurgaon,C002,null,silver,14/11/2023
null,Kolkata,C003,F,Silver,2022/09/13
null,BENGALURU,C004,Male,null,2017-02-21
19.0,Pune,C005,Male,PLATINUM,2022/06/06
37.0,Hyderabad,C006,f,null,27-Apr-2018
50.0,Kolkata,C007,M,null,2021/11/02
59.0,Bangalore,C008,null,Silver,17-Jun-2021
22.0,BENGALURU,C009,F,null,06-May-2020
58.0,Hyderabad,C010,M,Gold,12/12/2019


In [0]:
display(df_raw_orders.limit(5))

order_id,order_date,customer_id,customer_name,product_id,product_name,category,quantity,price,payment_type,order_status,returned
1001,2025-07-28,C004,Customer_4,P103,Denim Jean,Apparel,1,339.97,card,shipped,no
1002,2025-09-09,C054,Customer_54,P117,USB Cable,Accessories,1,$380.61,cash,completed,no
1003,2025-08-13,C059,Customer_59,P101,T-shirt XL,Apparel,3,278.26,card,completed,no
1004,11-Sep-2025,C013,Customer_13,P107,Charger,Accessories,3,$85.51,upi,completed,no
1005,24/08/2025,C035,Customer_35,P112,Backpack,Accessories,1,343.88,cash,cancelled,no


In [0]:
from pyspark.sql.functions import *

spark.conf.set("spark.sql.ansi.enabled", "false")

order_ts = coalesce(
    to_timestamp(trim(col("order_date")), "yyyy-MM-dd"),
    to_timestamp(trim(col("order_date")), "dd/MM/yyyy"),
    to_timestamp(trim(col("order_date")), "d-MMM-yyyy"),
    to_timestamp(trim(col("order_date")), "MM-dd-yyyy"),
    to_timestamp(trim(col("order_date")), "dd-MM-yyyy")
)

cleans_orders = (
    df_raw_orders 
    .withColumn("order_id", trim(col("order_id")).cast("long")) 
    .withColumn("order_date_ts", order_ts)
    .withColumn("customer_id", trim(col("customer_id"))) 
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("product_name", lower(trim(col("product_name"))))
    .withColumn("category", lower(trim(col("category"))))
    .withColumn("quantity", when(trim(col("quantity")).isNull() | (trim(col("quantity")) == ""), lit(1)).otherwise(trim(col("quantity"))))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("quantity", when(col("quantity") <= 0, lit(1)).otherwise(col("quantity")))
    .withColumn("price_clean", translate(trim(col("price")), "$ ", ""))
    .withColumn("price", col("price_clean").cast("double"))
    .drop("price_clean")
    .withColumn("payment_type", lower(trim(col("payment_type"))))
    .withColumn("order_status", lower(trim(col("order_status"))))
    .withColumn("returned", lower(trim(col("returned"))))
    .withColumn("total_amount", (coalesce(col("quantity"), lit(0)) * coalesce(col("price"), lit(0.0))).cast("double"))
)



cleans_orders = cleans_orders.dropDuplicates(["order_id", "product_id"]).filter(col("order_id").isNotNull() & col("product_id").isNotNull())

cleans_orders.write.format("delta").mode("overwrite").saveAsTable("projectcatalog.sales_data.silver_orders")

# Save as file
cleans_orders.write.format("parquet").mode("overwrite").save("/Volumes/projectcatalog/sales_data/data/silver/t_dw_silver_orders.csv")


display(cleans_orders)

order_id,order_date,customer_id,customer_name,product_id,product_name,category,quantity,price,payment_type,order_status,returned,order_date_ts,total_amount
1039,08-19-2025,C079,Customer_79,P108,t-shirt l,apparel,1,220.61,cash,cancelled,no,2025-08-19T00:00:00.000Z,220.61
1046,26/07/2025,C080,Customer_80,P118,jacket,apparel,3,314.63,card,shipped,yes,2025-07-26T00:00:00.000Z,943.89
1064,20-Jul-2025,C072,Customer_72,P112,backpack,accessories,3,270.6,cash,delivered,no,2025-07-20T00:00:00.000Z,811.8000000000001
1086,29-Jul-2025,C020,Customer_20,P119,sunglasses,accessories,3,158.56,card,delivered,no,2025-07-29T00:00:00.000Z,475.68
1112,08-12-2025,C056,Customer_56,P103,denim jean,apparel,3,186.14,cash,completed,no,2025-08-12T00:00:00.000Z,558.42
1124,07/08/2025,C024,Customer_24,P108,t-shirt l,apparel,3,91.87,cash,completed,no,2025-08-07T00:00:00.000Z,275.61
1127,10/09/2025,C061,Customer_61,P117,usb cable,accessories,3,382.12,cash,returned,yes,2025-09-10T00:00:00.000Z,1146.3600000000001
1145,09-14-2025,C030,Customer_30,P101,t-shirt xl,apparel,3,139.19,upi,completed,no,2025-09-14T00:00:00.000Z,417.57
1191,2025-07-28,C026,Customer_26,P102,sneakers,shoes,2,453.01,card,shipped,no,2025-07-28T00:00:00.000Z,906.02
1192,31-Jul-2025,C055,Customer_55,P112,backpack,accessories,1,143.53,upi,delivered,no,2025-07-31T00:00:00.000Z,143.53


Cleaning retail data

In [0]:
display(df_raw_customers.limit(5))

age,city,customer_id,gender,loyalty_tier,signup_date
35.0,Delhi,C001,M,Gold,01-Apr-2020
39.0,Gurgaon,C002,null,silver,14/11/2023
null,Kolkata,C003,F,Silver,2022/09/13
null,BENGALURU,C004,Male,null,2017-02-21
19.0,Pune,C005,Male,PLATINUM,2022/06/06


In [0]:
from pyspark.sql.functions import *

spark.conf.set("spark.sql.ansi.enabled", "false")

signup_ts = coalesce(
    to_timestamp(trim(col("signup_date")), "yyyy-MM-dd"),
    to_timestamp(trim(col("signup_date")), "dd/MM/yyyy"),
    to_timestamp(trim(col("signup_date")), "d-MMM-yyyy"),
    to_timestamp(trim(col("signup_date")), "yyyy/MM/dd")
)

clean_customers = (
    df_raw_customers
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("gender", lower(trim(col("gender"))))
    .withColumn("age", when(trim(col("age")).isNull() | (trim(col("age")) == ""), None).otherwise(col("age")))
    .withColumn("age", when(col("age").cast("int") < 0, None).otherwise(col("age").cast("int"))) 
    .withColumn("city", trim(col("city"))) 
    .withColumn("loyalty_tier", lower(trim(col("loyalty_tier")))) 
    .withColumn("signup_date", signup_ts)
)

display(clean_customers)

clean_customers.write.format("delta").mode("overwrite").saveAsTable("projectcatalog.sales_data.silver_customers")
cleans_orders.write.format("parquet").mode("overwrite").save("/Volumes/projectcatalog/sales_data/data/silver/t_dw_silver_customers.csv")



age,city,customer_id,gender,loyalty_tier,signup_date
35,Delhi,C001,m,gold,2020-04-01T00:00:00.000Z
39,Gurgaon,C002,null,silver,2023-11-14T00:00:00.000Z
null,Kolkata,C003,f,silver,2022-09-13T00:00:00.000Z
null,BENGALURU,C004,male,null,2017-02-21T00:00:00.000Z
19,Pune,C005,male,platinum,2022-06-06T00:00:00.000Z
37,Hyderabad,C006,f,null,2018-04-27T00:00:00.000Z
50,Kolkata,C007,m,null,2021-11-02T00:00:00.000Z
59,Bangalore,C008,null,silver,2021-06-17T00:00:00.000Z
22,BENGALURU,C009,f,null,2020-05-06T00:00:00.000Z
58,Hyderabad,C010,m,gold,2019-12-12T00:00:00.000Z


In [0]:
%sql
select * from projectcatalog.sales_data.silver_customers limit 100

age,city,customer_id,gender,loyalty_tier,signup_date
35,Delhi,C001,m,gold,2020-04-01T00:00:00.000Z
39,Gurgaon,C002,null,silver,2023-11-14T00:00:00.000Z
null,Kolkata,C003,f,silver,2022-09-13T00:00:00.000Z
null,BENGALURU,C004,male,null,2017-02-21T00:00:00.000Z
19,Pune,C005,male,platinum,2022-06-06T00:00:00.000Z
37,Hyderabad,C006,f,null,2018-04-27T00:00:00.000Z
50,Kolkata,C007,m,null,2021-11-02T00:00:00.000Z
59,Bangalore,C008,null,silver,2021-06-17T00:00:00.000Z
22,BENGALURU,C009,f,null,2020-05-06T00:00:00.000Z
58,Hyderabad,C010,m,gold,2019-12-12T00:00:00.000Z


In [0]:
silver_orders = spark.table("projectcatalog.sales_data.silver_customers")
silver_customers = spark.table("projectcatalog.sales_data.silver_orders")

orders_enriched = silver_orders.join(silver_customers, on="customer_id", how="left")

orders_enriched = orders_enriched.withColumn("order_date", to_date(col("order_date_ts")))\
                                .withColumn("year", year(col("order_date")))\
                                .withColumn("month", month(col("order_date")))

gold_df = (
    orders_enriched
    .groupBy("product_id", "product_name", "customer_id", "category", "year", "month", "order_date", "gender", "age", "city", "loyalty_tier")
    .agg(
        sum("total_amount").alias("total_sales"),
        sum(coalesce(col("quantity"), lit(0))).alias("total_quantity"),
        countDistinct("order_id").alias("total_orders"),
        min("price").alias("min_price"),
        max("price").alias("max_price"),
        avg("price").alias("avg_unit_price"),
        sum(when(col("returned") == "yes", col("total_amount")).otherwise(0.0)).alias("returned_amount")
    )
)


gold_df = gold_df.withColumn("avg_order_value", when(col("total_orders")>0, col("total_sales")/col("total_orders")).otherwise(lit(0.0))) \
                 .withColumn("avg_price_per_item", when(col("total_quantity")>0, col("total_sales")/col("total_quantity")).otherwise(lit(0.0))) \
                 .withColumn("refreshed_at", current_timestamp())

display(gold_df)

gold_df.write.format("delta").mode("overwrite").saveAsTable("projectcatalog.sales_data.gold_df")
gold_df.write.format("parquet").mode("overwrite").save("/Volumes/projectcatalog/sales_data/data/gold/t_dw_gold_df.csv")

product_id,product_name,customer_id,category,year,month,order_date,gender,age,city,loyalty_tier,total_sales,total_quantity,total_orders,min_price,max_price,avg_unit_price,returned_amount,avg_order_value,avg_price_per_item,refreshed_at
P101,t-shirt xl,C049,apparel,2025,8,2025-08-23,male,26,Delhi,null,298.99,1,1,298.99,298.99,298.99,0.0,298.99,298.99,2026-06-30T18:23:43.607Z
P101,t-shirt xl,C032,apparel,2025,8,2025-08-12,null,null,Pune,silver,140.62,1,1,140.62,140.62,140.62,0.0,140.62,140.62,2026-06-30T18:23:43.607Z
P111,socks,C011,apparel,2025,9,2025-09-17,m,21,delhi,gold,308.91,1,1,308.91,308.91,308.91,0.0,308.91,308.91,2026-06-30T18:23:43.607Z
P102,sneakers,C035,shoes,2025,8,2025-08-27,female,39,Mumbai,platinum,210.02,1,1,210.02,210.02,210.02,210.02,210.02,210.02,2026-06-30T18:23:43.607Z
P110,watch,C031,accessories,2025,8,2025-08-31,m,34,Lucknow,platinum,629.13,3,1,209.71,209.71,209.71,0.0,629.13,209.71,2026-06-30T18:23:43.607Z
P115,mouse,C030,electronics,2025,7,2025-07-22,null,39,null,bronze,261.35,1,1,261.35,261.35,261.35,0.0,261.35,261.35,2026-06-30T18:23:43.607Z
P117,usb cable,C054,accessories,2025,9,2025-09-09,f,26,delhi,gold,380.61,1,1,380.61,380.61,380.61,0.0,380.61,380.61,2026-06-30T18:23:43.607Z
P109,jeans,C025,apparel,2025,8,2025-08-22,m,23,mumbai,silver,309.52,1,1,309.52,309.52,309.52,0.0,309.52,309.52,2026-06-30T18:23:43.607Z
P117,usb cable,C036,accessories,2025,8,2025-08-23,m,48,Gurgaon,gold,448.4,1,1,448.4,448.4,448.4,0.0,448.4,448.4,2026-06-30T18:23:43.607Z
P107,charger,C012,accessories,2025,8,2025-08-28,female,52,Noida,platinum,474.29999999999995,3,1,158.1,158.1,158.1,0.0,474.29999999999995,158.1,2026-06-30T18:23:43.607Z
